# Homework: a second question for the same pipeline, worked solution

The finished script is `sessions/session-10/homework/solutions/pipeline.py`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-10/demos")
import genre_pipeline

plays = genre_pipeline.clean_plays(genre_pipeline.load_messy(genre_pipeline.MESSY))
OUTPUT = Path("output")
print(len(plays), "plays")

## 1. `minutes_by_month(plays)`

In [ ]:
def minutes_by_month(plays):
    """Return one row per month ("2024-01" and so on): plays and minutes."""
    month = plays["played_at"].str[:7]
    months = (plays.assign(month=month)
              .groupby("month")
              .agg(plays=("play_id", "count"), minutes=("minutes_played", "sum"))
              .reset_index())
    return months.round({"minutes": 1})


months = minutes_by_month(plays)
assert len(months) == 24
assert months["plays"].sum() == len(plays)
assert months.iloc[0]["month"] == "2024-01"
months.head()

`assign(month=...)` adds a column to a copy of the table, so `plays` itself
is left alone. A function that quietly changed the DataFrame it was given
would surprise every other stage that used it afterwards.

`played_at` is YYYY-MM-DD text, so `.str[:7]` is the session 4 `[:7]`
slice done to the whole column.

**Why year-first text sorts as dates:** `groupby` sorts the months as text,
and every month has two digits, so `"2024-02"` comes before `"2024-10"`.
This is the `played_at[:7]` lesson from the session 4 stretch. Written as
`"10-2024"`, January 2025 would sort before February 2024.

## 2. `save_month_chart(months, path)`

In [ ]:
def save_month_chart(months, path):
    """Draw minutes per month as a line chart, titled with the busiest month."""
    busiest = months.loc[months["minutes"].idxmax()]
    label = pd.to_datetime(busiest["month"]).strftime("%B %Y")

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(months["month"], months["minutes"], color="#4f6ddb", linewidth=2, marker="o")
    ax.set_title(f"{label} was my busiest month: {busiest['minutes']:.1f} minutes")
    ax.set_ylabel("minutes played")
    ax.set_ylim(bottom=0)                       # an honest axis starts at zero
    ax.tick_params(axis="x", labelrotation=90)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


save_month_chart(months, OUTPUT / "minutes_by_month.png")

In [ ]:
from IPython.display import Image

Image(filename="output/minutes_by_month.png", width=700)

**Line or bars?** The months have an order, and the question is about
change, so the eye should travel along a line. Genres have no order, so
they got bars. That is the bar-versus-line rule from session 9.

**What the line says:** a dip every summer. The three quietest months are
all in summer 2025, and 2024 dips in the same place.

In [ ]:
quietest = months.sort_values("minutes").head(3)
assert list(quietest["month"]) == ["2025-07", "2025-08", "2025-06"]
assert months.loc[months["minutes"].idxmax(), "month"] == "2024-10"
quietest

July 2025 is the quietest at 219.6 minutes; October 2024 the busiest at
531.2. Summer 2024 dips too: 288.7, 234.1 and 266.8 minutes for June, July
and August. The play counts in the session 4 stretch had the same shape,
so two different measures tell one story.

**What `set_ylim(bottom=0)` does:** without it, matplotlib starts the axis
a little below the quietest month, around 200. Then July 2025 sits on the
floor of the chart and looks like listening stopped, when it was really
about 40% of the busiest month. Starting at zero keeps the heights honest.

## 3. `check_months(path, plays)`

In [ ]:
def check_months(path, plays):
    """Read the monthly CSV back and confirm every play landed in a month."""
    back = pd.read_csv(path)
    assert len(back) == 24, "two years should be 24 months"
    assert back["plays"].sum() == len(plays), "plays went missing on the way"


genre_pipeline.save_csv(months, OUTPUT / "minutes_by_month.csv")
check_months(OUTPUT / "minutes_by_month.csv", plays)
print("the monthly file agrees with the data")

`save_csv` came from the genre pipeline and was used unchanged on a
completely different table. That is the payoff of naming its parameter
for what it is, a table, rather than for the one table it was written for.

## 4. The script

The solution script adds the three functions and these lines to `main()`.
Nothing above them changed:

```python
months = minutes_by_month(plays)
month_csv = OUTPUT / "minutes_by_month.csv"
month_png = OUTPUT / "minutes_by_month.png"
save_csv(months, month_csv)
save_month_chart(months, month_png)
check_months(month_csv, plays)
```

In [ ]:
import subprocess

result = subprocess.run([sys.executable, "sessions/session-10/homework/solutions/pipeline.py"],
                        capture_output=True, text=True, check=True)
print(result.stdout)

## Stretch: which year had more listening?

In [ ]:
by_year = months.groupby(months["month"].str[:4])["minutes"].sum().round(1)
print(by_year)
print(f"2024 had {by_year['2024'] - by_year['2025']:.1f} more minutes than 2025")
assert by_year["2024"] > by_year["2025"]

2024 wins by about 234 minutes, and you never had to go back to `plays`.
`months["month"].str[:4]` is the session 4 text slice, done to a whole
column at once.

One caution: each month was rounded to one decimal before being added up,
so a total built from the monthly file can be a few hundredths away from
one built from the raw plays. For a headline that is fine. For a check, add
up the unrounded numbers, or allow for it, which is what the `< 1` in
today's `check_summary` does.